
# Australia monthly weather: a complete (but simple) modelling study

This notebook is designed to be:

- **understandable** (good undergrad level: functions + plain dataframes, no classes),
- **thesis‑friendly** (clear assumptions, diagnostics, backtests),
- **robust to missing data** (explicit imputation + flags),
- focused on **two regimes**:
  - the **bulk / usual** behaviour, and
  - the **extremes** via **EVT (POT + GPD)**.

We work from one file:

- `monthly_data.csv`

and we produce:

- EDA + missingness study
- naive baselines + an ARIMA‑style baseline
- bulk quantile climatology model
- EVT tail model + threshold sensitivity + bootstrap uncertainty
- rolling backtests (time)
- spatial leave‑one‑station‑out evaluation (map credibility)
- forecast tables + maps
- an `index.html` that links everything in the output folder

---

## Mathematical summary (high level)

Let \(y_{s,t}\) be the monthly value at station \(s\) and month \(t\).

### Bulk model (climatological quantiles)
For each station and month‑of‑year \(m\in\{1,\dots,12\}\), estimate

\[
Q_{s,m}(q) = \text{empirical quantile of } y_{s,t} \text{ over training months with } \text{month}(t)=m.
\]

This gives a **median** forecast \(Q_{s,m}(0.5)\) and an **IQR band** \([Q_{s,m}(0.25),Q_{s,m}(0.75)]\).

### Naive baselines
- **Persistence:** \(\hat y_{s,t} = y_{s,t_0}\) (last available value before the forecast origin).
- **Seasonal naive (12‑lag):** \(\hat y_{s,t} = y_{s,t-12}\).
- **Pooled median:** \(\hat y_{s,t} = \text{median of training data}\).

### ARIMA‑style baseline (simple SARIMA idea)
We remove a monthly climatology and do a **seasonal difference**:

\[
a_{s,t}=y_{s,t}-Q_{s,\text{month}(t)}(0.5),\qquad
w_{s,t}=a_{s,t}-a_{s,t-12}.
\]

Then fit a simple AR(1)

\[
w_{s,t} = c_s + \phi_s w_{s,t-1} + \varepsilon_{s,t}.
\]

### EVT tail model (POT + GPD on standardised residuals)
Let \(\mu_{s,t}=Q_{s,m}(0.5)\) and \(\sigma_{s,t}\) be a scale proxy (from the IQR). Define residuals

\[
r_{s,t} =
\begin{cases}
\displaystyle \frac{y_{s,t}-\mu_{s,t}}{\sigma_{s,t}} & \text{upper tail} \\\\[6pt]
\displaystyle \frac{\mu_{s,t}-y_{s,t}}{\sigma_{s,t}} & \text{lower tail}.
\end{cases}
\]

Choose a high threshold \(u_s\) (e.g. the 95th percentile of \(r_{s,t}\) on training data). For exceedances
\[
x_{s,t} = r_{s,t}-u_s \;\; \big| \;\; r_{s,t}>u_s,
\]
fit a **Generalised Pareto Distribution** (GPD) with shape \(\xi_s\) and scale \(\beta_s\). This yields extreme quantiles (e.g. 0.99) and allows **threshold sensitivity** and **bootstrap CIs**.

---

> **Plotting note:** This notebook uses Plotly (HTML) to avoid the common NumPy‑2 / matplotlib binary mismatch on macOS. Every plot is also **saved** into the output folder.


In [ ]:

# If you need these packages (uncomment once):
# !pip install -U plotly kaleido

import os, json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook"  # plots show inside the notebook

np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.max_columns", 200)


## Configuration

Change only the next cell (target variable + tail).

In [ ]:

# ----------------------------
# USER CONFIG (edit this cell)
# ----------------------------
DATA_PATH = "monthly_data.csv"      # put the CSV next to this notebook
OUTPUT_DIR = "outputs_evt_study"    # everything saved here

TARGET = "rain"     # "rain" | "sun" | "lowest_max" | "highest_min"
TAIL   = "upper"    # "upper" for high extremes, "lower" for low extremes (cold extremes)

# Backtest settings (simple rolling-origin)
FOLD_MONTHS = 12      # length of each test fold
N_FOLDS     = 5       # number of folds
MIN_TRAIN_MONTHS = 10*12

# Forecast horizon (months ahead)
HORIZON = 12

# EVT settings
P0_LIST = [0.90, 0.925, 0.95, 0.975]   # threshold quantiles for sensitivity study
P0_MAIN = 0.95                         # main threshold used in backtests/forecast
EXTREME_ALPHA = 0.01                   # tail probability (0.01 -> 0.99 upper quantile, or 0.01 lower quantile)

SIGMA_FLOOR = 0.10  # floor for the residual scale proxy (in transformed units)

# Spatial CV (map credibility)
SPATIAL_EVAL_MONTHS = 10*12   # last 10 years
IDW_POWER = 2.0

# Plot export
SAVE_PNG = True   # requires kaleido; if it fails, HTML is still saved

# Example station for detailed EVT/stationarity diagnostics (set None for auto choice)
EXAMPLE_STATION = None


## Helper: saving plots (show + save)

In [ ]:

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name, show=True, save_png=SAVE_PNG):
    # Save HTML (always) and PNG (if kaleido works). Optionally show.
    html_path = out_dir / f"{name}.html"
    fig.write_html(str(html_path), include_plotlyjs="cdn")
    if save_png:
        try:
            png_path = out_dir / f"{name}.png"
            fig.write_image(str(png_path), scale=2)
        except Exception as e:
            print(f"[PNG export skipped for {name}] {type(e).__name__}: {e}")
    if show:
        fig.show()
    return html_path

def year_month_to_date(df):
    return pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))

def clip_physical(y, target):
    y = np.asarray(y, dtype=float)
    if target == "rain":
        y = np.maximum(y, 0.0)
    return y



## Load and reshape the data

We:
1. read `monthly_data.csv`,
2. standardise column names,
3. create a `date` column (month start),
4. build a **complete monthly grid per station**,
5. keep all four variables, but model only `TARGET`.


In [ ]:

raw = pd.read_csv(r"/Users/sk/Downloads/monthly_data.csv")

raw = raw.rename(columns={
    "Station number":"station",
    "Lat":"lat_raw",
    "Long":"lon",
    "Alt":"alt",
    "East/West":"east_west",
    "Coastal":"coastal",
    "Year":"year",
    "Month":"month",
    "Lowest max":"lowest_max",
    "Highest min":"highest_min",
})

raw["date"] = year_month_to_date(raw)

# latitude is stored as positive degrees SOUTH in this dataset
raw["lat"] = -np.abs(raw["lat_raw"].astype(float))

raw = raw.sort_values(["station","date"]).reset_index(drop=True)

meta_cols = ["station","lat","lon","alt","east_west","coastal"]
station_meta = raw[meta_cols].drop_duplicates("station").set_index("station")

# complete monthly grid per station
grids = []
for st, sub in raw.groupby("station"):
    dmin, dmax = sub["date"].min(), sub["date"].max()
    g = pd.DataFrame({"station": st, "date": pd.date_range(dmin, dmax, freq="MS")})
    grids.append(g)

grid = pd.concat(grids, ignore_index=True)

panel = grid.merge(raw[["station","date","rain","sun","lowest_max","highest_min"]],
                   on=["station","date"], how="left")

panel["month"] = panel["date"].dt.month
panel = panel.merge(station_meta.reset_index(), on="station", how="left")

panel.head()



## EDA: coverage and missingness (all variables)

We create:
- a station coverage table
- missingness heatmaps (one per variable)
- a station map (points)
- missingness by month-of-year (is missingness seasonal?)


In [ ]:

vars_all = ["rain","sun","lowest_max","highest_min"]

cov_rows = []
for st, sub in panel.groupby("station"):
    row = {"station": int(st), "lat": float(sub["lat"].iloc[0]), "lon": float(sub["lon"].iloc[0])}
    for v in vars_all:
        obs = sub[v].notna()
        row[f"{v}_n"] = int(obs.sum())
        row[f"{v}_pct_obs"] = float(obs.mean())
        row[f"{v}_first_obs"] = str(sub.loc[obs, "date"].min()) if obs.any() else ""
        row[f"{v}_last_obs"]  = str(sub.loc[obs, "date"].max()) if obs.any() else ""
    cov_rows.append(row)

coverage = pd.DataFrame(cov_rows).sort_values("station")
coverage.to_csv(out_dir / "station_coverage_all_vars.csv", index=False)
coverage.head()


In [33]:

def missingness_matrix(df, var):
    sub = df[["station","date",var]].copy()
    stations = sorted(sub["station"].unique())
    sub = sub[sub["station"].isin(stations)]
    mat = sub.pivot_table(index="station", columns="date", values=var, aggfunc="first")
    miss = mat.isna().astype(int)
    return miss

def plot_missingness(miss_mat, title):
    fig = go.Figure(data=go.Heatmap(
        z=miss_mat.values,
        x=miss_mat.columns,
        y=miss_mat.index.astype(str),
        colorscale="viridis",
        showscale=False
    ))
    fig.update_layout(title=title, xaxis_title="Date", yaxis_title="Station")
    return fig

for v in vars_all:
    miss = missingness_matrix(panel, v)
    fig = plot_missingness(miss, f"Missingness: {v} (1=missing)")
    save_fig(fig, f"eda_missingness_{v}", show=True)


[PNG export skipped for eda_missingness_rain] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for eda_missingness_sun] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for eda_missingness_lowest_max] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for eda_missingness_highest_min] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



In [8]:

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=coverage["lon"], lat=coverage["lat"],
    mode="markers+text",
    text=coverage["station"].astype(str),
    textposition="top center",
    marker=dict(size=8),
))
fig.update_geos(scope="world", lataxis_range=[-45, -10], lonaxis_range=[110, 155],
                showcountries=True, showland=True)
fig.update_layout(title="Stations (Australia window)")
save_fig(fig, "eda_station_map", show=True)


[PNG export skipped for eda_station_map] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



PosixPath('outputs_evt_study/eda_station_map.html')

In [9]:

rows = []
for v in vars_all:
    tmp = panel.groupby("month")[v].apply(lambda s: float(s.isna().mean())).reset_index(name="miss_rate")
    tmp["var"] = v
    rows.append(tmp)
miss_by_month = pd.concat(rows, ignore_index=True)
miss_by_month.to_csv(out_dir / "missingness_by_month_all_vars.csv", index=False)

fig = go.Figure()
for v in vars_all:
    sub = miss_by_month[miss_by_month["var"]==v]
    fig.add_trace(go.Scatter(x=sub["month"], y=sub["miss_rate"], mode="lines+markers", name=v))
fig.update_layout(title="Missingness rate by month-of-year", xaxis_title="Month", yaxis_title="Fraction missing")
save_fig(fig, "eda_missingness_by_month_all_vars", show=True)


[PNG export skipped for eda_missingness_by_month_all_vars] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



PosixPath('outputs_evt_study/eda_missingness_by_month_all_vars.html')


## Basic QC / outlier flags (rule-based)

We **do not delete** values; we only flag them to understand risk for EVT.


In [10]:

def qc_flags(df, var):
    s = df[var]
    flags = pd.Series(False, index=df.index)

    if var in ["rain","sun"]:
        flags |= (s < 0)

    if var in ["lowest_max","highest_min"]:
        flags |= (s < -30) | (s > 60)

    med = s.median(skipna=True)
    iqr = s.quantile(0.75) - s.quantile(0.25)
    if pd.notna(iqr) and iqr > 0:
        flags |= (s > med + 8*iqr)
    return flags

qc_rows = []
for v in vars_all:
    flags = qc_flags(panel, v)
    bad = panel.loc[flags, ["station","date",v]].copy()
    bad = bad.rename(columns={v:"value"})
    bad["var"] = v
    qc_rows.append(bad)

qc_table = pd.concat(qc_rows, ignore_index=True)
qc_table.to_csv(out_dir / "qc_flags_all_vars.csv", index=False)

print("Flagged points:", len(qc_table))
qc_table.head(10)


Flagged points: 59


,station,date,value,var
0,14015,1942-01-01,550.9,rain
1,14015,1949-02-01,553.4,rain
2,14015,1950-01-01,593.0,rain
3,14015,1955-02-01,654.0,rain
4,14015,1956-02-01,629.5,rain
5,14015,1957-01-01,564.0,rain
6,14015,1960-01-01,568.1,rain
7,14015,1962-01-01,571.0,rain
8,14015,1965-03-01,595.0,rain
9,14015,1965-12-01,582.9,rain



## Missingness handling (used in modelling)

For a chosen `TARGET`, define:

- `y_obs`: observed value (can be NaN)
- `y_model`: filled value (simple training-only imputation)

Imputation rule (training-only, to avoid leakage):

1. station × month median
2. station median
3. global median


In [20]:

def impute_train_test(train_df, test_df, ycol):
    train = train_df.copy()
    test  = test_df.copy()

    train["y_obs"] = train[ycol]
    test["y_obs"]  = test[ycol]

    med_sm = train.groupby(["station","month"])["y_obs"].median()
    med_s  = train.groupby("station")["y_obs"].median()
    med_g  = float(train["y_obs"].median())

    def fill_df(df):
        y = df["y_obs"].copy()
        key = list(zip(df["station"], df["month"]))
        y1 = y.fillna(pd.Series([med_sm.get(k, np.nan) for k in key], index=df.index))
        y2 = y1.fillna(df["station"].map(med_s))
        y3 = y2.fillna(med_g)
        out = df.copy()
        out["y_model"] = y3
        out["is_imputed"] = (out["y_obs"].isna() & out["y_model"].notna()).astype(int)
        return out

    return fill_df(train), fill_df(test)



## Transformations

- rain: \(x = \log(1+y)\)
- others: \(x=y\)

All modelling is in \(x\), then we invert back to \(y\).


In [11]:

def fwd_transform(y, target):
    y = np.asarray(y, dtype=float)
    if target == "rain":
        return np.log1p(np.maximum(y, 0.0))
    return y

def inv_transform(x, target):
    x = np.asarray(x, dtype=float)
    if target == "rain":
        return np.expm1(x)
    return x



## Naive baselines (explicit)

- persistence: last training value
- seasonal naive: 12-month lag
- pooled median: global median


In [12]:

def naive_persistence(train, test, ycol="y_model"):
    last = train.sort_values("date").groupby("station")[ycol].last()
    out = test[["station","date"]].copy()
    out["pred_persist"] = out["station"].map(last)
    return out

def naive_seasonal12(train, test, ycol="y_model"):
    train_map = train.set_index(["station","date"])[ycol]
    med_s = train.groupby("station")[ycol].median()

    out = test[["station","date"]].copy()
    lag_dates = out["date"] - pd.DateOffset(years=1)

    preds = []
    for st, d in zip(out["station"], lag_dates):
        val = train_map.get((st, d), np.nan)
        if pd.isna(val):
            val = med_s.get(st, np.nan)
        preds.append(val)
    out["pred_seasonal12"] = preds
    return out

def naive_pooled_median(train, test, ycol="y_model"):
    gmed = float(train[ycol].median())
    out = test[["station","date"]].copy()
    out["pred_pooled"] = gmed
    return out



## Bulk model: station × month quantiles

We compute \(q_{0.25}, q_{0.5}, q_{0.75}\) in transformed space \(x\).


In [13]:

def fit_bulk_quantiles(train, xcol="x_model"):
    qtbl = (train.groupby(["station","month"])[xcol]
            .quantile([0.25,0.5,0.75])
            .unstack()
            .reset_index()
            .rename(columns={0.25:"q25",0.5:"q50",0.75:"q75"}))
    return qtbl

def predict_bulk(df, qtbl):
    out = df.merge(qtbl, on=["station","month"], how="left")
    if out[["q25","q50","q75"]].isna().any().any():
        station_q = qtbl.groupby("station")[["q25","q50","q75"]].median().reset_index()
        out = out.drop(columns=["q25","q50","q75"]).merge(station_q, on="station", how="left")
    return out

def sigma_from_iqr(q25, q75, floor=SIGMA_FLOOR):
    iqr = np.asarray(q75) - np.asarray(q25)
    sigma = iqr / 1.349
    sigma = np.maximum(sigma, floor)
    return sigma



## ARIMA-style baseline (simple SARIMA idea)

We implement a seasonal AR(1) model on seasonal differences of deseasonalised anomalies.


In [14]:

def fit_ar1_ols(w):
    w = np.asarray(w, dtype=float)
    w = w[np.isfinite(w)]
    if len(w) < 20:
        return np.nan, np.nan
    y = w[1:]
    x = w[:-1]
    X = np.column_stack([np.ones_like(x), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    c, phi = float(beta[0]), float(beta[1])
    return c, phi

def predict_sar1(train, test, target, xcol="x_model"):
    med_sm = train.groupby(["station","month"])[xcol].median()

    train = train.copy()
    train["a"] = [r[xcol] - med_sm.get((r["station"], r["month"]), np.nan) for _, r in train.iterrows()]

    params = {}
    for st, sub in train.groupby("station"):
        sub = sub.sort_values("date")
        a = sub["a"].values
        if len(a) < 40:
            continue
        w = a[12:] - a[:-12]
        c, phi = fit_ar1_ols(w)
        if not np.isfinite(phi):
            continue
        params[st] = (c, phi, a[-12:], w[-1])

    out = test[["station","date","month"]].copy()
    out["pred_arima"] = np.nan

    for st, sub in out.groupby("station"):
        if st not in params:
            continue
        c, phi, last_a, last_w = params[st]
        sub = sub.sort_values("date").copy()
        h = len(sub)

        w_hat = np.zeros(h)
        w_prev = last_w
        for i in range(h):
            w_prev = c + phi*w_prev
            w_hat[i] = w_prev

        a_hat = np.zeros(h)
        for i in range(h):
            a_ref = last_a[i] if i < 12 else a_hat[i-12]
            a_hat[i] = w_hat[i] + a_ref

        x_hat = []
        for m, a_i in zip(sub["month"].values, a_hat):
            x_hat.append(med_sm.get((st, m), np.nan) + a_i)
        x_hat = np.array(x_hat, dtype=float)

        y_hat = inv_transform(x_hat, target)
        y_hat = clip_physical(y_hat, target)

        out.loc[sub.index, "pred_arima"] = y_hat

    return out[["station","date","pred_arima"]]



## EVT: POT + GPD

We fit a GPD to exceedances over a high threshold in **standardised residual space**.
We also produce:
- threshold sensitivity (\(\xi\) vs threshold),
- bootstrap return level curve.


In [15]:

def gpd_fit_lmom(exceedances):
    x = np.asarray(exceedances, dtype=float)
    x = x[np.isfinite(x)]
    x = x[x >= 0]
    n = len(x)
    if n < 30:
        return np.nan, np.nan

    x = np.sort(x)
    i = np.arange(1, n+1)
    w = (i-1)/(n-1) if n > 1 else np.zeros(n)

    b0 = x.mean()
    b1 = np.mean(w*x)
    L1 = b0
    L2 = 2*b1 - b0

    if not np.isfinite(L1) or not np.isfinite(L2) or L1 <= 0 or L2 <= 0:
        return np.nan, np.nan

    t = L2 / L1
    xi = 2 - 1/t
    beta = L1 * (1 - xi)

    if not np.isfinite(beta) or beta <= 0:
        return np.nan, np.nan
    return float(xi), float(beta)

def gpd_quantile_from_pot(alpha, p_u, u, xi, beta):
    if not (np.isfinite(xi) and np.isfinite(beta) and np.isfinite(u) and np.isfinite(p_u)):
        return np.nan
    if p_u <= 0 or alpha <= 0:
        return np.nan
    if abs(xi) < 1e-9:
        y = beta * np.log(p_u/alpha)
    else:
        y = (beta/xi) * ((p_u/alpha)**xi - 1.0)
    return float(u + y)

def fit_evt_per_station(train, mu_col="q50", q25_col="q25", q75_col="q75", xcol="x_model",
                        tail="upper", p0=P0_MAIN):
    rows = []
    for st, sub in train.groupby("station"):
        sigma = sigma_from_iqr(sub[q25_col].values, sub[q75_col].values, floor=SIGMA_FLOOR)
        mu = sub[mu_col].values
        x  = sub[xcol].values

        if tail == "upper":
            r = (x - mu) / sigma
        else:
            r = (mu - x) / sigma

        r = r[np.isfinite(r)]
        if len(r) < 100:
            rows.append({"station": st, "u": np.nan, "p_u": np.nan, "xi": np.nan, "beta": np.nan,
                         "r_q_emp": np.nan, "n_exc": 0, "n": int(len(r))})
            continue

        u = float(np.quantile(r, p0))
        exc = r[r > u] - u
        p_u = float(np.mean(r > u))
        xi, beta = gpd_fit_lmom(exc)
        q_upper = 1 - EXTREME_ALPHA
        r_q_emp = float(np.quantile(r, q_upper))

        rows.append({"station": st, "u": u, "p_u": p_u, "xi": xi, "beta": beta,
                     "r_q_emp": r_q_emp, "n_exc": int(len(exc)), "n": int(len(r))})
    return pd.DataFrame(rows)

def apply_extremes(df, evt_tbl, tail="upper"):
    out = df.copy().merge(evt_tbl, on="station", how="left")

    sigma = sigma_from_iqr(out["q25"].values, out["q75"].values, floor=SIGMA_FLOOR)
    mu = out["q50"].values

    alpha = EXTREME_ALPHA
    q_upper = 1 - alpha

    r_q_emp = out["r_q_emp"].values
    x_ext_bulk = np.where(np.isfinite(r_q_emp),
                          mu + sigma*r_q_emp if tail=="upper" else mu - sigma*r_q_emp,
                          np.nan)

    r_q_evt = np.array([gpd_quantile_from_pot(alpha, p_u, u, xi, beta)
                        for (p_u, u, xi, beta) in zip(out["p_u"], out["u"], out["xi"], out["beta"])], dtype=float)

    x_ext_evt = np.where(np.isfinite(r_q_evt),
                         mu + sigma*r_q_evt if tail=="upper" else mu - sigma*r_q_evt,
                         np.nan)

    y_ext_bulk = clip_physical(inv_transform(x_ext_bulk, TARGET), TARGET)
    y_ext_evt  = clip_physical(inv_transform(x_ext_evt,  TARGET), TARGET)

    out["pred_ext_bulk"] = y_ext_bulk
    out["pred_ext_evt"]  = y_ext_evt
    return out



## Stationarity, trend, change-points

We compute:
- ADF-style t-stat on deseasonalised series
- OLS trend slope on annual means
- Pettitt change-point on annual means


In [16]:

def adf_tstat(z):
    z = np.asarray(z, dtype=float)
    z = z[np.isfinite(z)]
    if len(z) < 50:
        return np.nan
    dz = z[1:] - z[:-1]
    lag = z[:-1]
    X = np.column_stack([np.ones_like(lag), lag])
    beta, *_ = np.linalg.lstsq(X, dz, rcond=None)
    resid = dz - X@beta
    n, k = X.shape
    s2 = np.sum(resid**2) / (n - k)
    cov = s2 * np.linalg.inv(X.T@X)
    se_gamma = np.sqrt(cov[1,1])
    gamma = beta[1]
    return float(gamma / se_gamma) if se_gamma > 0 else np.nan

def pettitt_test(x, dates):
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(x)
    x = x[mask]
    dates = np.asarray(dates)[mask]
    n = len(x)
    if n < 30:
        return None, None, np.nan, np.nan

    r = pd.Series(x).rank(method="average").values
    cum = np.cumsum(r)
    t = np.arange(1, n+1)
    U = 2*cum - t*(n+1)
    K = np.max(np.abs(U))
    tau = int(np.argmax(np.abs(U)))
    p = 2*np.exp((-6*K*K)/(n**3 + n**2))
    return tau, pd.Timestamp(dates[tau]), float(min(p,1.0)), float(K)

def annual_mean_series(df, ycol):
    tmp = df.dropna(subset=[ycol]).copy()
    tmp["year"] = tmp["date"].dt.year
    ann = tmp.groupby("year")[ycol].mean().reset_index()
    ann["date"] = pd.to_datetime(dict(year=ann["year"], month=1, day=1))
    return ann

study_rows = []
for st, sub in panel.groupby("station"):
    y = sub[TARGET]
    if y.notna().sum() < 60:
        continue

    sm = sub.groupby("month")[TARGET].median()
    y_fill = y.fillna(sub["month"].map(sm)).fillna(y.median())

    x_fill = fwd_transform(y_fill.values, TARGET)
    mmed = pd.Series(x_fill).groupby(sub["month"].values).transform("median").values
    a = x_fill - mmed

    adf_raw = adf_tstat(a)
    adf_seas = adf_tstat(a[12:] - a[:-12]) if len(a) > 24 else np.nan

    ann = annual_mean_series(pd.DataFrame({"date": sub["date"], "y": y_fill}), "y")
    if len(ann) >= 10:
        years = ann["year"].values.astype(float)
        vals  = ann["y"].values.astype(float)
        slope = np.polyfit(years, vals, 1)[0]
    else:
        slope = np.nan

    cp_i, cp_date, cp_p, cp_K = pettitt_test(ann["y"].values, ann["date"].values) if len(ann) >= 10 else (None,None,np.nan,np.nan)

    study_rows.append({
        "station": int(st),
        "lat": float(sub["lat"].iloc[0]),
        "lon": float(sub["lon"].iloc[0]),
        "adf_tstat_deseason": adf_raw,
        "adf_tstat_seasdiff": adf_seas,
        "annual_trend_slope_per_year": slope,
        "pettitt_cp_date": str(cp_date) if cp_date is not None else "",
        "pettitt_p": cp_p,
        "pettitt_K": cp_K,
    })

stationarity_tbl = pd.DataFrame(study_rows).sort_values("station")
stationarity_tbl.to_csv(out_dir / f"stationarity_trend_changepoint_{TARGET}.csv", index=False)
stationarity_tbl.head()


,station,lat,lon,adf_tstat_deseason,adf_tstat_seasdiff,annual_trend_slope_per_year,pettitt_cp_date,pettitt_p,pettitt_K
0,9225,-31.92,115.87,-19.167834,-18.941518,-0.348429,2005-01-01 00:00:00,0.470657,112.0
1,13011,-26.13,126.58,-27.166241,-27.892367,0.044817,1997-01-01 00:00:00,0.004967,816.0
2,14015,-12.42,130.89,-28.046543,-27.197253,0.617049,1967-01-01 00:00:00,0.021007,699.0
3,23034,-34.95,138.52,-25.144108,-26.122235,-0.118099,2001-01-01 00:00:00,0.123643,419.0
4,23373,-34.48,139.01,-19.751464,-21.613300,-0.116435,2017-01-01 00:00:00,0.520935,108.0


In [17]:

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=stationarity_tbl["lon"], lat=stationarity_tbl["lat"],
    mode="markers+text",
    text=stationarity_tbl["station"].astype(str),
    textposition="top center",
    marker=dict(size=10, color=stationarity_tbl["annual_trend_slope_per_year"],
                colorbar=dict(title="Slope / year"), showscale=True)
))
fig.update_geos(scope="world", lataxis_range=[-45, -10], lonaxis_range=[110, 155],
                showland=True, showcountries=True)
fig.update_layout(title=f"Annual trend slope map ({TARGET})")
save_fig(fig, f"stationarity_trend_map_{TARGET}", show=True)

cp_years = stationarity_tbl["pettitt_cp_date"].replace("", np.nan).dropna().apply(lambda s: pd.to_datetime(s).year)
fig = go.Figure(data=go.Histogram(x=cp_years))
fig.update_layout(title=f"Pettitt change-point years across stations ({TARGET})",
                  xaxis_title="Year", yaxis_title="Count")
save_fig(fig, f"stationarity_changepoint_years_{TARGET}", show=True)


[PNG export skipped for stationarity_trend_map_rain] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for stationarity_changepoint_years_rain] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



PosixPath('outputs_evt_study/stationarity_changepoint_years_rain.html')


## Rolling-origin backtest (time)

Each fold:
- trains on all data before the fold,
- tests on the next `FOLD_MONTHS` months,
- evaluates only on months with observed `y_obs`.


In [18]:

last_obs_date = panel.dropna(subset=[TARGET])["date"].max()
fold_ends = [last_obs_date - pd.DateOffset(months=FOLD_MONTHS*i) for i in range(N_FOLDS)][::-1]
fold_starts = [d - pd.DateOffset(months=FOLD_MONTHS-1) for d in fold_ends]

folds = []
data_start = panel["date"].min()
for fs, fe in zip(fold_starts, fold_ends):
    test_start = fs
    test_end = fe
    train_end = test_start - pd.DateOffset(months=1)
    n_train = (train_end.year - data_start.year)*12 + (train_end.month - data_start.month) + 1
    if n_train >= MIN_TRAIN_MONTHS:
        folds.append((train_end, test_start, test_end))

print("Folds used:", len(folds))
folds


Folds used: 5


[(Timestamp('2021-01-01 00:00:00'),
  Timestamp('2021-02-01 00:00:00'),
  Timestamp('2022-01-01 00:00:00')),
 (Timestamp('2022-01-01 00:00:00'),
  Timestamp('2022-02-01 00:00:00'),
  Timestamp('2023-01-01 00:00:00')),
 (Timestamp('2023-01-01 00:00:00'),
  Timestamp('2023-02-01 00:00:00'),
  Timestamp('2024-01-01 00:00:00')),
 (Timestamp('2024-01-01 00:00:00'),
  Timestamp('2024-02-01 00:00:00'),
  Timestamp('2025-01-01 00:00:00')),
 (Timestamp('2025-01-01 00:00:00'),
  Timestamp('2025-02-01 00:00:00'),
  Timestamp('2026-01-01 00:00:00'))]

In [21]:

def pinball_loss(y, qhat, q):
    y = np.asarray(y, dtype=float)
    qhat = np.asarray(qhat, dtype=float)
    mask = np.isfinite(y) & np.isfinite(qhat)
    y = y[mask]; qhat = qhat[mask]
    if len(y) == 0:
        return np.nan
    return float(np.mean(np.maximum(q*(y-qhat), (q-1)*(y-qhat))))

def run_one_fold(train_end, test_start, test_end):
    train = panel[panel["date"] <= train_end].copy()
    test  = panel[(panel["date"] >= test_start) & (panel["date"] <= test_end)].copy()

    keep = ["station","date","month","lat","lon",TARGET]
    train = train[keep]
    test  = test[keep]

    train, test = impute_train_test(train, test, TARGET)

    train["x_model"] = fwd_transform(train["y_model"].values, TARGET)
    test["x_model"]  = fwd_transform(test["y_model"].values, TARGET)

    qtbl = fit_bulk_quantiles(train, xcol="x_model")
    train_b = predict_bulk(train, qtbl)
    test_b  = predict_bulk(test, qtbl)

    test_b["pred_bulk_median"] = clip_physical(inv_transform(test_b["q50"].values, TARGET), TARGET)
    test_b["pred_bulk_q25"]    = clip_physical(inv_transform(test_b["q25"].values, TARGET), TARGET)
    test_b["pred_bulk_q75"]    = clip_physical(inv_transform(test_b["q75"].values, TARGET), TARGET)

    test_b = test_b.merge(naive_persistence(train, test, ycol="y_model"), on=["station","date"], how="left")
    test_b = test_b.merge(naive_seasonal12(train, test, ycol="y_model"), on=["station","date"], how="left")
    test_b = test_b.merge(naive_pooled_median(train, test, ycol="y_model"), on=["station","date"], how="left")

    test_b["pred_climo"] = test_b["pred_bulk_median"]

    arima_pred = predict_sar1(train_b, test_b[["station","date","month"]].copy(), TARGET, xcol="x_model")
    test_b = test_b.merge(arima_pred, on=["station","date"], how="left")

    evt_tbl = fit_evt_per_station(train_b, xcol="x_model", tail=TAIL, p0=P0_MAIN)
    test_e = apply_extremes(test_b, evt_tbl, tail=TAIL)

    eval_df = test_e.dropna(subset=["y_obs"]).copy()

    thr = train.groupby("station")["y_obs"].quantile(1-EXTREME_ALPHA if TAIL=="upper" else EXTREME_ALPHA)
    eval_df["thr_station"] = eval_df["station"].map(thr)

    if TAIL=="upper":
        eval_df["is_extreme_month"] = (eval_df["y_obs"] >= eval_df["thr_station"]).astype(int)
        exceed_evt = np.mean(eval_df["y_obs"] > eval_df["pred_ext_evt"])
        exceed_bulk = np.mean(eval_df["y_obs"] > eval_df["pred_ext_bulk"])
        extreme_mask = eval_df["is_extreme_month"].values == 1
    else:
        eval_df["is_extreme_month"] = (eval_df["y_obs"] <= eval_df["thr_station"]).astype(int)
        exceed_evt = np.mean(eval_df["y_obs"] < eval_df["pred_ext_evt"])
        exceed_bulk = np.mean(eval_df["y_obs"] < eval_df["pred_ext_bulk"])
        extreme_mask = eval_df["is_extreme_month"].values == 1

    expected = EXTREME_ALPHA

    def mae(a,b):
        a=np.asarray(a); b=np.asarray(b)
        mask = np.isfinite(a) & np.isfinite(b)
        if mask.sum()==0: return np.nan
        return float(np.mean(np.abs(a[mask]-b[mask])))

    def rmse(a,b):
        a=np.asarray(a); b=np.asarray(b)
        mask = np.isfinite(a) & np.isfinite(b)
        if mask.sum()==0: return np.nan
        return float(np.sqrt(np.mean((a[mask]-b[mask])**2)))

    y = eval_df["y_obs"].values
    preds = {
        "persist": eval_df["pred_persist"].values,
        "seasonal12": eval_df["pred_seasonal12"].values,
        "pooled": eval_df["pred_pooled"].values,
        "climo": eval_df["pred_climo"].values,
        "arima": eval_df["pred_arima"].values,
        "bulk": eval_df["pred_bulk_median"].values,
    }

    metrics = []
    for name, phat in preds.items():
        metrics.append({
            "model": name,
            "mae": mae(y, phat),
            "rmse": rmse(y, phat),
            "mae_usual": mae(y[~extreme_mask], phat[~extreme_mask]),
            "mae_extreme": mae(y[extreme_mask], phat[extreme_mask]),
        })

    cover = np.mean((y >= eval_df["pred_bulk_q25"].values) & (y <= eval_df["pred_bulk_q75"].values))

    q_eval = 1-EXTREME_ALPHA if TAIL=="upper" else EXTREME_ALPHA
    pb_bulk_ext = pinball_loss(y, eval_df["pred_ext_bulk"].values, q_eval)
    pb_evt_ext  = pinball_loss(y, eval_df["pred_ext_evt"].values,  q_eval)

    fold_summary = {
        "train_end": str(train_end.date()),
        "test_start": str(test_start.date()),
        "test_end": str(test_end.date()),
        "n_eval": int(len(eval_df)),
        "bulk_iqr_coverage": float(cover),
        "ext_exceed_bulk": float(exceed_bulk),
        "ext_exceed_evt": float(exceed_evt),
        "ext_exceed_expected": float(expected),
        "pinball_ext_bulk": pb_bulk_ext,
        "pinball_ext_evt": pb_evt_ext,
    }

    return fold_summary, pd.DataFrame(metrics), eval_df, evt_tbl

fold_summaries = []
fold_metrics = []
pred_frames = []
evt_frames = []

for i,(train_end, test_start, test_end) in enumerate(folds, start=1):
    fs, mt, preds, evt_tbl = run_one_fold(train_end, test_start, test_end)
    fs["fold"] = i
    mt["fold"] = i
    fold_summaries.append(fs)
    fold_metrics.append(mt)
    preds["fold"] = i
    pred_frames.append(preds)
    evt_tbl["fold"] = i
    evt_frames.append(evt_tbl)

backtest_summary = pd.DataFrame(fold_summaries)
backtest_metrics = pd.concat(fold_metrics, ignore_index=True)
backtest_preds   = pd.concat(pred_frames, ignore_index=True)
evt_params       = pd.concat(evt_frames, ignore_index=True)

backtest_summary.to_csv(out_dir / f"backtest_summary_{TARGET}_{TAIL}.csv", index=False)
backtest_metrics.to_csv(out_dir / f"backtest_metrics_{TARGET}_{TAIL}.csv", index=False)
backtest_preds.to_csv(out_dir / f"backtest_predictions_{TARGET}_{TAIL}.csv", index=False)
evt_params.to_csv(out_dir / f"backtest_evt_params_{TARGET}_{TAIL}.csv", index=False)

backtest_summary


,train_end,test_start,test_end,n_eval,bulk_iqr_coverage,ext_exceed_bulk,ext_exceed_evt,ext_exceed_expected,pinball_ext_bulk,pinball_ext_evt,fold
0,2021-01-01,2021-02-01,2022-01-01,202,0.455446,0.019802,0.004950,0.01,4.012468e+08,7.341493e+07,1
1,2022-01-01,2022-02-01,2023-01-01,197,0.390863,0.081218,0.055838,0.01,1.432970e+08,3.013913e+07,2
2,2023-01-01,2023-02-01,2024-01-01,192,0.447917,0.010417,0.015625,0.01,4.984526e+07,3.982095e+07,3
3,2024-01-01,2024-02-01,2025-01-01,190,0.478947,0.000000,0.000000,0.01,1.788657e+07,3.800543e+07,4
4,2025-01-01,2025-02-01,2026-01-01,188,0.531915,0.005319,0.005319,0.01,9.818532e+06,3.477535e+07,5


In [22]:

avg_mae = (backtest_metrics.groupby("model")[["mae","mae_usual","mae_extreme"]]
           .mean()
           .reset_index()
           .sort_values("mae"))

fig = go.Figure()
fig.add_trace(go.Bar(x=avg_mae["model"], y=avg_mae["mae"], name="MAE (all)"))
fig.add_trace(go.Bar(x=avg_mae["model"], y=avg_mae["mae_usual"], name="MAE (usual)"))
fig.add_trace(go.Bar(x=avg_mae["model"], y=avg_mae["mae_extreme"], name="MAE (extreme months)"))
fig.update_layout(barmode="group", title=f"Backtest MAE by model ({TARGET}, {TAIL})", yaxis_title="MAE")
save_fig(fig, f"plot_backtest_mae_{TARGET}_{TAIL}", show=True)

fig = go.Figure()
fig.add_trace(go.Scatter(x=backtest_summary["fold"], y=backtest_summary["ext_exceed_bulk"], mode="lines+markers", name="Bulk extreme"))
fig.add_trace(go.Scatter(x=backtest_summary["fold"], y=backtest_summary["ext_exceed_evt"], mode="lines+markers", name="EVT extreme"))
fig.add_trace(go.Scatter(x=backtest_summary["fold"], y=backtest_summary["ext_exceed_expected"], mode="lines", name="Expected"))
fig.update_layout(title=f"Extreme calibration by fold (expected ≈ {EXTREME_ALPHA})",
                  xaxis_title="Fold", yaxis_title="Exceedance rate")
save_fig(fig, f"plot_extreme_calibration_{TARGET}_{TAIL}", show=True)

if EXAMPLE_STATION is None:
    counts = backtest_preds.groupby("station")["y_obs"].count().sort_values(ascending=False)
    EXAMPLE_STATION = int(counts.index[0])

sub = backtest_preds[backtest_preds["station"]==EXAMPLE_STATION].sort_values("date").copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=sub["date"], y=sub["y_obs"], mode="lines", name="Observed"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_persist"], mode="lines", name="Persistence"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_seasonal12"], mode="lines", name="Seasonal naive"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_climo"], mode="lines", name="Climatology"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_arima"], mode="lines", name="ARIMA-style"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_bulk_median"], mode="lines", name="Bulk median"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_bulk_q75"], mode="lines", name="Bulk q75", line=dict(dash="dot")))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_bulk_q25"], mode="lines", name="Bulk q25", line=dict(dash="dot"), fill="tonexty"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_ext_bulk"], mode="lines", name="Bulk extreme", line=dict(dash="dash")))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_ext_evt"], mode="lines", name="EVT extreme", line=dict(dash="dash")))
fig.update_layout(title=f"Backtest predictions (station {EXAMPLE_STATION})", yaxis_title=TARGET)
save_fig(fig, f"plot_backtest_station_{TARGET}_{TAIL}_station{EXAMPLE_STATION}", show=True)


[PNG export skipped for plot_backtest_mae_rain_upper] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for plot_extreme_calibration_rain_upper] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for plot_backtest_station_rain_upper_station14015] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



PosixPath('outputs_evt_study/plot_backtest_station_rain_upper_station14015.html')


## Spatial leave-one-station-out evaluation (map credibility)

We interpolate each month using IDW from other stations' observed values.


In [23]:

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1,lon1,lat2,lon2])
    dlat = lat2-lat1
    dlon = lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def idw_predict(lats, lons, vals, lat0, lon0, power=2.0):
    lats = np.asarray(lats); lons = np.asarray(lons); vals = np.asarray(vals)
    mask = np.isfinite(vals)
    lats=lats[mask]; lons=lons[mask]; vals=vals[mask]
    if len(vals) < 3:
        return np.nan
    d = haversine_km(lats, lons, lat0, lon0)
    d = np.maximum(d, 1e-6)
    w = 1.0/(d**power)
    return float(np.sum(w*vals)/np.sum(w))

eval_end = last_obs_date
eval_start = eval_end - pd.DateOffset(months=SPATIAL_EVAL_MONTHS-1)
eval_panel = panel[(panel["date"] >= eval_start) & (panel["date"] <= eval_end)].copy()
pre_panel = panel[panel["date"] < eval_start].copy()

thr_station = pre_panel.groupby("station")[TARGET].quantile(1-EXTREME_ALPHA if TAIL=="upper" else EXTREME_ALPHA)

rows=[]
stations = sorted(panel["station"].unique())

for hold in stations:
    lat0 = float(station_meta.loc[hold,"lat"])
    lon0 = float(station_meta.loc[hold,"lon"])

    sub_hold = eval_panel[eval_panel["station"]==hold].copy()
    thr = thr_station.get(hold, np.nan)

    obs=[]
    preds=[]
    is_ext=[]

    for d, y0 in zip(sub_hold["date"].values, sub_hold[TARGET].values):
        if not np.isfinite(y0):
            continue
        others = eval_panel[(eval_panel["date"]==d) & (eval_panel["station"]!=hold)][["lat","lon",TARGET]].dropna()
        if len(others) < 3:
            continue
        yhat = idw_predict(others["lat"], others["lon"], others[TARGET], lat0, lon0, power=IDW_POWER)
        if not np.isfinite(yhat):
            continue

        obs.append(float(y0))
        preds.append(float(yhat))
        if np.isfinite(thr):
            is_ext.append(int(y0 >= thr) if TAIL=="upper" else int(y0 <= thr))
        else:
            is_ext.append(0)

    obs=np.array(obs); preds=np.array(preds); is_ext=np.array(is_ext)

    def mae(a,b):
        if len(a)==0: return np.nan
        return float(np.mean(np.abs(a-b)))
    def rmse(a,b):
        if len(a)==0: return np.nan
        return float(np.sqrt(np.mean((a-b)**2)))

    rows.append({
        "station": int(hold),
        "n": int(len(obs)),
        "mae": mae(obs,preds),
        "rmse": rmse(obs,preds),
        "mae_usual": mae(obs[is_ext==0], preds[is_ext==0]),
        "mae_extreme": mae(obs[is_ext==1], preds[is_ext==1]),
        "lat": lat0, "lon": lon0
    })

spatial_cv = pd.DataFrame(rows).sort_values("mae")
spatial_cv.to_csv(out_dir / f"spatial_cv_idw_{TARGET}_{TAIL}.csv", index=False)
spatial_cv.head()


,station,n,mae,rmse,mae_usual,mae_extreme,lat,lon
3,23034,120,10.266464,15.663706,10.266464,NaN,-34.95,138.52
4,23373,118,11.273876,16.523199,10.122708,55.401975,-34.48,139.01
5,24024,119,15.606392,19.249158,15.427024,22.541942,-34.44,140.60
13,76031,120,15.921895,18.915460,15.921895,NaN,-34.24,142.09
11,65034,120,24.006565,31.361647,24.006565,NaN,-32.56,148.95


In [24]:

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=spatial_cv["lon"], lat=spatial_cv["lat"],
    mode="markers+text",
    text=spatial_cv["station"].astype(str),
    textposition="top center",
    marker=dict(size=10, color=spatial_cv["mae"], colorbar=dict(title="MAE"), showscale=True)
))
fig.update_geos(scope="world", lataxis_range=[-45,-10], lonaxis_range=[110,155], showland=True, showcountries=True)
fig.update_layout(title=f"Spatial leave-one-out (IDW) MAE map ({TARGET})")
save_fig(fig, f"plot_spatial_cv_mae_map_{TARGET}_{TAIL}", show=True)

fig = go.Figure()
fig.add_trace(go.Bar(x=spatial_cv["station"].astype(str), y=spatial_cv["mae_usual"], name="MAE usual"))
fig.add_trace(go.Bar(x=spatial_cv["station"].astype(str), y=spatial_cv["mae_extreme"], name="MAE extreme"))
fig.update_layout(barmode="group", title=f"Spatial IDW errors by station ({TARGET})",
                  xaxis_title="Station", yaxis_title="MAE")
save_fig(fig, f"plot_spatial_cv_usual_extreme_{TARGET}_{TAIL}", show=True)


[PNG export skipped for plot_spatial_cv_mae_map_rain_upper] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for plot_spatial_cv_usual_extreme_rain_upper] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



PosixPath('outputs_evt_study/plot_spatial_cv_usual_extreme_rain_upper.html')


## Fit on all data and forecast forward

We forecast the next `HORIZON` months and save a forecast table + plots.


In [25]:

full = panel[["station","date","month","lat","lon",TARGET]].copy()

# full-history imputation (final fit)
sm = full.groupby(["station","month"])[TARGET].median()
smed = full.groupby("station")[TARGET].median()
gmed = float(full[TARGET].median())

full["y_obs"] = full[TARGET]
key = list(zip(full["station"], full["month"]))
fill1 = full["y_obs"].fillna(pd.Series([sm.get(k, np.nan) for k in key], index=full.index))
fill2 = fill1.fillna(full["station"].map(smed))
fill3 = fill2.fillna(gmed)
full["y_model"] = fill3
full["is_imputed"] = (full["y_obs"].isna() & full["y_model"].notna()).astype(int)

full["x_model"] = fwd_transform(full["y_model"].values, TARGET)

qtbl_full = fit_bulk_quantiles(full, xcol="x_model")

last_date = last_obs_date
future_dates = pd.date_range(last_date + pd.DateOffset(months=1), periods=HORIZON, freq="MS")
future = pd.DataFrame([(st,d) for st in sorted(full["station"].unique()) for d in future_dates],
                      columns=["station","date"])
future["month"] = future["date"].dt.month
future = future.merge(station_meta.reset_index(), on="station", how="left")

hist_map = full.set_index(["station","date"])["y_model"]

future_b = predict_bulk(future, qtbl_full)
future_b["pred_bulk_median"] = clip_physical(inv_transform(future_b["q50"].values, TARGET), TARGET)
future_b["pred_bulk_q25"]    = clip_physical(inv_transform(future_b["q25"].values, TARGET), TARGET)
future_b["pred_bulk_q75"]    = clip_physical(inv_transform(future_b["q75"].values, TARGET), TARGET)

last_y = full.sort_values("date").groupby("station")["y_model"].last()
future_b["pred_persist"] = future_b["station"].map(last_y)

future_b["pred_seasonal12"] = np.nan
for st in future_b["station"].unique():
    idx = future_b["station"]==st
    lag = future_b.loc[idx, "date"] - pd.DateOffset(years=1)
    vals = [hist_map.get((st, d), np.nan) for d in lag]
    if np.all(pd.isna(vals)):
        vals = [float(smed.get(st, gmed))]*len(vals)
    future_b.loc[idx, "pred_seasonal12"] = vals

future_b["pred_pooled"] = gmed
future_b["pred_climo"] = future_b["pred_bulk_median"]

# ARIMA forecast
full_for_arima = full.merge(predict_bulk(full, qtbl_full)[["station","date","month","q25","q50","q75"]],
                            on=["station","date","month"], how="left")
arima_future = predict_sar1(full_for_arima, future_b[["station","date","month"]].copy(), TARGET, xcol="x_model")
future_b = future_b.merge(arima_future, on=["station","date"], how="left")

# EVT fit on full
evt_full = fit_evt_per_station(full_for_arima, xcol="x_model", tail=TAIL, p0=P0_MAIN)
future_e = apply_extremes(future_b, evt_full, tail=TAIL)

forecast_tbl = future_e.copy()
forecast_tbl.to_csv(out_dir / f"forecast_table_{TARGET}_{TAIL}.csv", index=False)
forecast_tbl.head()


,station,date,month,lat,lon,alt,east_west,coastal,q25,q50,q75,pred_bulk_median,pred_bulk_q25,pred_bulk_q75,pred_persist,pred_seasonal12,pred_pooled,pred_climo,pred_arima,u,p_u,xi,beta,r_q_emp,n_exc,n,pred_ext_bulk,pred_ext_evt
0,9225,2026-02-01,2,-31.92,115.87,25,W,1,0.182322,1.386294,2.484907,3.0,0.200000,11.000000,3.0,0.0,40.25,3.0,4.861124,1.586148,0.050691,NaN,NaN,2.284529,22,434,196.495988,NaN
1,9225,2026-03-01,3,-31.92,115.87,25,W,1,1.774511,2.721295,3.420946,14.2,4.897398,29.598347,3.0,19.0,40.25,14.2,4.934475,1.586148,0.050691,NaN,NaN,2.284529,22,434,246.034945,NaN
2,9225,2026-04-01,4,-31.92,115.87,25,W,1,2.723764,3.273364,3.817648,25.4,14.237564,44.497059,3.0,9.2,40.25,25.4,18.522325,1.586148,0.050691,NaN,NaN,2.284529,22,434,167.317941,NaN
3,9225,2026-05-01,5,-31.92,115.87,25,W,1,4.145489,4.454347,4.658943,85.0,62.148508,104.524530,3.0,62.4,40.25,85.0,85.216759,1.586148,0.050691,NaN,NaN,2.284529,22,434,204.178667,NaN
4,9225,2026-06-01,6,-31.92,115.87,25,W,1,4.549657,4.803201,5.027340,120.9,93.600000,151.526690,3.0,120.9,40.25,120.9,121.202036,1.586148,0.050691,NaN,NaN,2.284529,22,434,272.733469,NaN


In [26]:

st = EXAMPLE_STATION
sub = forecast_tbl[forecast_tbl["station"]==st].sort_values("date").copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_persist"], mode="lines", name="Persistence"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_seasonal12"], mode="lines", name="Seasonal naive"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_climo"], mode="lines", name="Climatology"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_arima"], mode="lines", name="ARIMA-style"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_bulk_median"], mode="lines", name="Bulk median"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_bulk_q75"], mode="lines", name="Bulk q75", line=dict(dash="dot")))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_bulk_q25"], mode="lines", name="Bulk q25", line=dict(dash="dot"), fill="tonexty"))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_ext_bulk"], mode="lines", name="Bulk extreme", line=dict(dash="dash")))
fig.add_trace(go.Scatter(x=sub["date"], y=sub["pred_ext_evt"], mode="lines", name="EVT extreme", line=dict(dash="dash")))
fig.update_layout(title=f"Forecasts (station {st})", yaxis_title=TARGET)
save_fig(fig, f"plot_forecast_station_{TARGET}_{TAIL}_station{st}", show=True)

last_f = forecast_tbl["date"].max()
snap = forecast_tbl[forecast_tbl["date"]==last_f].copy()

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=snap["lon"], lat=snap["lat"],
    mode="markers+text",
    text=snap["station"].astype(str),
    textposition="top center",
    marker=dict(size=10, color=snap["pred_bulk_median"], colorbar=dict(title="Median"), showscale=True)
))
fig.update_geos(scope="world", lataxis_range=[-45,-10], lonaxis_range=[110,155], showland=True, showcountries=True)
fig.update_layout(title=f"Forecast median at stations ({TARGET}) - {last_f.date()}")
save_fig(fig, f"map_forecast_median_points_{TARGET}_{TAIL}_{last_f.date()}", show=True)

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lon=snap["lon"], lat=snap["lat"],
    mode="markers+text",
    text=snap["station"].astype(str),
    textposition="top center",
    marker=dict(size=10, color=snap["pred_ext_evt"], colorbar=dict(title="Extreme"), showscale=True)
))
fig.update_geos(scope="world", lataxis_range=[-45,-10], lonaxis_range=[110,155], showland=True, showcountries=True)
fig.update_layout(title=f"Forecast EVT extreme at stations ({TARGET}) - {last_f.date()}")
save_fig(fig, f"map_forecast_extreme_points_{TARGET}_{TAIL}_{last_f.date()}", show=True)


[PNG export skipped for plot_forecast_station_rain_upper_station14015] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for map_forecast_median_points_rain_upper_2027-01-01] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for map_forecast_extreme_points_rain_upper_2027-01-01] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



PosixPath('outputs_evt_study/map_forecast_extreme_points_rain_upper_2027-01-01.html')


### Optional: IDW surface maps (visual only)

We interpolate station predictions to a grid (IDW). These are for visualisation, not a physical model.


In [27]:

def idw_grid_surface(snap, value_col, power=2.0, n_lat=120, n_lon=140):
    lat_min, lat_max = -45, -10
    lon_min, lon_max = 110, 155
    lats = np.linspace(lat_min, lat_max, n_lat)
    lons = np.linspace(lon_min, lon_max, n_lon)
    Z = np.full((n_lat, n_lon), np.nan)

    pts_lat = snap["lat"].values
    pts_lon = snap["lon"].values
    pts_val = snap[value_col].values

    for i, la in enumerate(lats):
        for j, lo in enumerate(lons):
            Z[i,j] = idw_predict(pts_lat, pts_lon, pts_val, la, lo, power=power)
    return lats, lons, Z

for col, name in [("pred_bulk_median","median"), ("pred_ext_evt","extreme_evt")]:
    lats, lons, Z = idw_grid_surface(snap, col, power=IDW_POWER)
    fig = go.Figure(data=go.Contour(z=Z, x=lons, y=lats, contours_coloring="heatmap",
                                    colorbar=dict(title=name)))
    fig.update_layout(title=f"IDW surface ({name}) - {TARGET} - {last_f.date()}",
                      xaxis_title="Longitude", yaxis_title="Latitude")
    save_fig(fig, f"idw_surface_{name}_{TARGET}_{TAIL}_{last_f.date()}", show=False)

print("IDW surfaces saved.")


[PNG export skipped for idw_surface_median_rain_upper_2027-01-01] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

[PNG export skipped for idw_surface_extreme_evt_rain_upper_2027-01-01] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido

IDW surfaces saved.



## EVT diagnostics: threshold sensitivity + bootstrap return levels (example station)

We produce:
- \(\xi\) vs threshold quantile,
- exceedance counts vs threshold,
- bootstrap return level curve.


In [28]:

st = EXAMPLE_STATION

full_b = full_for_arima.copy()
sub = full_b[full_b["station"]==st].sort_values("date").copy()

sigma = sigma_from_iqr(sub["q25"].values, sub["q75"].values, floor=SIGMA_FLOOR)
mu = sub["q50"].values
x  = sub["x_model"].values

if TAIL=="upper":
    r = (x - mu)/sigma
else:
    r = (mu - x)/sigma
r = r[np.isfinite(r)]

sens_rows=[]
for p0 in P0_LIST:
    if len(r) < 200:
        continue
    u = float(np.quantile(r, p0))
    exc = r[r>u] - u
    xi,beta = gpd_fit_lmom(exc)
    sens_rows.append({"p0":p0, "u":u, "n_exc":int(len(exc)), "xi":xi, "beta":beta})
sens = pd.DataFrame(sens_rows)
sens.to_csv(out_dir / f"evt_threshold_sensitivity_station{st}_{TARGET}_{TAIL}.csv", index=False)

fig = go.Figure()
fig.add_trace(go.Scatter(x=sens["p0"], y=sens["xi"], mode="lines+markers", name="xi"))
fig.update_layout(title=f"Threshold sensitivity: xi vs threshold quantile (station {st})",
                  xaxis_title="P0", yaxis_title="xi")
save_fig(fig, f"evt_threshold_sensitivity_xi_station{st}_{TARGET}_{TAIL}", show=True)

fig = go.Figure()
fig.add_trace(go.Bar(x=sens["p0"], y=sens["n_exc"], name="#exceedances"))
fig.update_layout(title=f"Threshold sensitivity: exceedance counts (station {st})",
                  xaxis_title="P0", yaxis_title="# exceedances")
save_fig(fig, f"evt_threshold_sensitivity_counts_station{st}_{TARGET}_{TAIL}", show=True)

# Bootstrap return levels
u = float(np.quantile(r, P0_MAIN))
exc = r[r>u] - u
p_u = float(np.mean(r>u))
xi0,beta0 = gpd_fit_lmom(exc)

return_years = np.array([2,5,10,20], dtype=float)
alpha_T = 1/(12*return_years)
r_T = np.array([gpd_quantile_from_pot(a, p_u, u, xi0, beta0) for a in alpha_T])

B = 300
rng = np.random.default_rng(0)
exc = np.asarray(exc, dtype=float)
boot = []
for b in range(B):
    sample = exc[rng.integers(0, len(exc), size=len(exc))]
    xi,beta = gpd_fit_lmom(sample)
    if not np.isfinite(xi):
        continue
    boot.append([gpd_quantile_from_pot(a, p_u, u, xi, beta) for a in alpha_T])
boot = np.array(boot, dtype=float)

ci_lo = np.nanpercentile(boot, 2.5, axis=0)
ci_hi = np.nanpercentile(boot, 97.5, axis=0)

mu0 = float(np.nanmedian(mu))
sigma0 = float(np.nanmedian(sigma))

if TAIL=="upper":
    xT = mu0 + sigma0*r_T
    x_lo = mu0 + sigma0*ci_lo
    x_hi = mu0 + sigma0*ci_hi
else:
    xT = mu0 - sigma0*r_T
    x_lo = mu0 - sigma0*ci_hi
    x_hi = mu0 - sigma0*ci_lo

yT = clip_physical(inv_transform(xT, TARGET), TARGET)
y_lo = clip_physical(inv_transform(x_lo, TARGET), TARGET)
y_hi = clip_physical(inv_transform(x_hi, TARGET), TARGET)

ret_tbl = pd.DataFrame({"return_years":return_years, "return_level":yT, "ci_lo":y_lo, "ci_hi":y_hi})
ret_tbl.to_csv(out_dir / f"evt_return_levels_bootstrap_station{st}_{TARGET}_{TAIL}.csv", index=False)

fig = go.Figure()
fig.add_trace(go.Scatter(x=return_years, y=yT, mode="lines+markers", name="Return level"))
fig.add_trace(go.Scatter(x=return_years, y=y_hi, mode="lines", name="CI high", line=dict(dash="dot")))
fig.add_trace(go.Scatter(x=return_years, y=y_lo, mode="lines", name="CI low", line=dict(dash="dot"), fill="tonexty"))
fig.update_layout(title=f"Bootstrap return levels (station {st})", xaxis_title="Return period (years)", yaxis_title=TARGET)
save_fig(fig, f"evt_return_levels_bootstrap_station{st}_{TARGET}_{TAIL}", show=True)

ret_tbl


[PNG export skipped for evt_threshold_sensitivity_xi_station14015_rain_upper] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for evt_threshold_sensitivity_counts_station14015_rain_upper] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



[PNG export skipped for evt_return_levels_bootstrap_station14015_rain_upper] ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido



,return_years,return_level,ci_lo,ci_hi
0,2.0,453.150562,379.836246,568.351143
1,5.0,5689.750090,2567.805657,14805.843594
2,10.0,32801.896407,11236.184026,101365.867729
3,20.0,166467.761332,39057.311399,744890.439680



## Build `index.html` and zip the output folder

This is helpful for sharing and thesis appendices.


In [29]:

files = sorted([p.name for p in out_dir.iterdir() if p.is_file()])

sections = {
    "CSV tables": [f for f in files if f.endswith(".csv")],
    "HTML plots": [f for f in files if f.endswith(".html")],
    "PNG figures": [f for f in files if f.endswith(".png")],
}

html = []
html.append("<h1>Australia weather EVT study outputs</h1>")
html.append(f"<p><b>TARGET</b>: {TARGET} &nbsp; <b>TAIL</b>: {TAIL} &nbsp; <b>Generated</b>: {pd.Timestamp.now()}</p>")

for sec, flist in sections.items():
    html.append(f"<h2>{sec}</h2><ul>")
    for f in flist:
        html.append(f'<li><a href="{f}">{f}</a></li>')
    html.append("</ul>")

index_path = out_dir / "index.html"
index_path.write_text("\n".join(html), encoding="utf-8")
print("Wrote", index_path)

zip_path = out_dir.with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in out_dir.iterdir():
        if p.is_file():
            z.write(p, arcname=p.name)
print("Wrote", zip_path)


Wrote outputs_evt_study/index.html
Wrote outputs_evt_study.zip
